In [8]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
%reload_ext autoreload

In [10]:
from dotenv import load_dotenv
#from langchain_openai import ChatOpenAI
from importlib import reload
from tree_of_thought import TreeOfThought
import openai
import os, torch, numpy as np
from utils import call_llm
from plot_tree import plot_tree
import json
from datasets import load_dataset
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report

In [11]:
load_dotenv()

ENV_VARS = {
    "API_KEY_OPENAI": "OpenAI",
    #"API_KEY_MISTRAL": "Mistral"
}

for var, name in ENV_VARS.items():
    if not os.getenv(var):
        raise ValueError(f"Missing {name} API key: `{var}` must be set in the environment.")


client = openai.OpenAI(api_key= os.getenv("API_KEY_OPENAI"))
model = "gpt-4o-mini"

In [12]:
from stereotype_def import stereotype_definition

tot = TreeOfThought(client, model=model, max_branching_factor=3, max_depth=3, stereotype_definition=stereotype_definition)
initial_prompt = (
    #"Russians are born in Russia."
    #"Everyone I know in ethiopia is starving."
    #"French people like mangas and comics."
    #"The Bolivian man at the mall is handsome."
    #"Japanese man dressed well."
    "The policeman stated that Cody was the suspect in the crime."
)

solution = tot.solve(initial_prompt)
print("\n🧠 Full Tree Structure:\n")
tot.print_full_tree()

print("🧠 Reasoning Path:\n")
for i, thought in enumerate(solution):
    print(f"Level {i}:")
    print(f"  Thought            : {thought.content}")
    print(f"  Stereotypes/Biases : {thought.stereotypes_biases}")
    print(f"  Description        : {thought.description}")
    print(f"  Is Stereotype      : {thought.is_stereotyping}")
    print(f"  Score              : {thought.score:.2f}")
    print("-" * 70)

print("\n📊 Metrics Summary:")
print(f"  Total LLM Calls        : {tot.total_calls}")
print(f"  Total Tokens Used      : {tot.total_tokens}")
print(f"       Prompt Tokens     : {tot.total_prompt_tokens}")
print(f"       Completion Tokens : {tot.total_completion_tokens}")
print(f"  Total Latency (sec)    : {tot.total_latency:.2f}")
if tot.total_calls > 0:
    print(f"  Avg Latency per Call   : {tot.total_latency / tot.total_calls:.2f} sec")



🧠 Full Tree Structure:

🧠 Thought ID 0
→ Score: 0.00
→ Content: The policeman stated that Cody was the suspect in the crime.
------------------------------------------------------------
    🧠 Thought ID 0.1
    → Score: 0.80
    → Content: The assumption that Cody is the suspect may reflect biases based on his appearance or background.
    → Stereotypes: Racial or socio-economic profiling.
    → Description: This reasoning examines how certain identities or situations can lead to presumption of guilt without evidence, reflecting an overgeneralized association between appearance and criminality.
    → Is Stereotyping: Yes
    ------------------------------------------------------------
        🧠 Thought ID 0.1.1
        → Score: 0.80
        → Content: The assumption that Cody is a suspect due to his appearance may stem from existing societal biases regarding race or ethnicity.
        → Stereotypes: Racial profiling.
        → Description: This reasoning indicates how societal prejudi